In [1]:
import dill
import os
script_dir = os.getcwd()
parent_dir = os.path.dirname(script_dir)
print(parent_dir)

/home/vmanso/Victor/Tissue Characterization Paper/TFM-Vertex-Model


In [ ]:
noise = 0.25
pct = 0.25
sim = 0
type = 'slow_random_0.55'

if type == 'random':
    folder_dir = 'resultados'
elif type == 'cluster':
    folder_dir = 'resultados_cluster'
elif type == 'slow_random':
    folder_dir = 'resultados_lambda_0.2_random'
elif type == 'slow_cluster':
    folder_dir = 'resultados_lambda_0.2_cluster'
elif type == 'slow_random_0.55':
    folder_dir = 'resultados_lambda_0.55_random'
elif type == 'slow_cluster_0.55':
    folder_dir = 'resultados_lambda_0.55_cluster'

else:
    raise ValueError("Invalid type. Must be 'random', 'cluster', 'slow_random', or 'slow_cluster'.")

ValueError: Invalid type. Must be 'random', 'cluster', 'slow_random', or 'slow_cluster'.

In [ ]:
with open(os.path.join(parent_dir, folder_dir, f"noise_{noise}",f"pct_{pct}",f"sim_{sim}" ,"history.dill"), "rb") as f:
    history = dill.load(f)
import vertex_lite as model
from vertex_lite import coloring 

from vertex_lite import custom 


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation, rc
from IPython.display import HTML
import numpy as np  

In [ ]:
 
if type == 'random':
    import vertex_lite as model
    from vertex_lite.custom import shape_index
    from vertex_lite.forces import TargetArea, Tension, Perimeter, Pressure
    import vertex_lite.initialisation as init
    from vertex_lite.run_select import run, basic_simulation
    from vertex_lite import custom , coloring
    import math
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import gc
    import dill
    dt= 0.001
    viscosity = 0.02
    P=0.0
    K=1.0 #area elasticity
    G= 0.04 #contractility of the cell
    L=0.075 #line tensions
    Lambda_0 = 0.68 #lambda for the mutants
    t_end = 5
    rand=np.random.RandomState(123) #random seed for reproducibility
    N_cell_across= 20
    N_cell_up= 20
    N_total= N_cell_across * N_cell_up

    params=[K,G,L]
    N_Step = int(t_end / dt)
    skip = 5
    mesh_type =  0.25
    mutant_percentage = 0.25
    mesh = init.toroidal_hex_mesh(20, N_cell_across, noise=mesh_type, rand=rand)

    N_mutants = int(mutant_percentage * N_total)
    cells = model.Cells(mesh, properties={
        'K': K,
        'Gamma': G,
        'P': 0.0,
        'boundary_P': P,
        'Lambda': L,
        'Lambda_boundary': 0.5,
        'A0': 1.0
    })

    # Fase de relajación inicial
    step_init = int(50 / dt)
    history_init = run(basic_simulation(cells, TargetArea() + Tension() + Perimeter() + Pressure()), step_init, int(1 / dt))
    cells = history_init[-1].copy()
    del history_init
    gc.collect()
    # Mutación
    ids_mutant = rand.choice(N_total, size=N_mutants, replace=False)
    cells.properties['parent_group'] = np.zeros(len(cells), dtype=int)
    cells.properties['parent_group'][ids_mutant] = 1
    cells.properties['Gamma'] = np.array([G, 0])[cells.properties['parent_group']]
    cells.properties['Lambda'] = np.array([L, Lambda_0])[cells.properties['parent_group']]

    # Simulación principal

    history = run(basic_simulation(cells, TargetArea() + Tension() + Perimeter() + Pressure()), N_Step, skip)


In [ ]:

for cells in history:
    cells.properties['mut'] =coloring.definecolors_manual(cells, property_name='parent_group')
    cells.properties['shape_index'] = [custom.shape_index(cells,i, normalize=False, max_shape_index=5.0) for i in range(len(cells.mesh.area))]
    cells.properties['n_lados'] = np.array([len(cells.mesh.boundary(i)) for i in range(len(cells))])
    cells.properties['si'] =coloring.values_to_hex_colors2(cells.properties['shape_index'], cmap_name='custom', fixed_vmin=3.72, fixed_vmax=3.9)
    cells.properties['nl'] = coloring.definecolors_fixed_1_10_center6(cells, property_name='n_lados', cmap_name='BrBG')
for tocolor in ['mut', 'si', 'nl']:
    hh=history if 'slow' in type else history[0:1000]
    # print(len(hh))
    # print(len(hh)*5)
    fig=plt.figure()
    fig.set_size_inches(6, 6)
    ax=fig.gca()

    def init_fig():
        ax=plt.figure()
        return(ax,)

    def animate_fig(i):
        cells_array=hh
        v_max=np.max((np.max(cells_array[-1].mesh.vertices),np.max(cells_array[0].mesh.vertices)))
        size = 2.0 * v_max
        cells = hh[i]
        time = int(5*i)
        model.plotting.drawnoplt_videos(cells,ax,size,tocolor=tocolor)
        ax.set_title(f"Time: {time}")
        return ax,

    anim = animation.FuncAnimation(fig, animate_fig, frames=int(len(hh)), init_func=init_fig)
    slow='_slow' if 'slow' in type else ''
    anim.save(f'{type}{slow}_{tocolor}.mp4', fps=30, extra_args=['-vcodec', 'libx264'],writer='ffmpeg')
    HTML(anim.to_html5_video())
    

KeyboardInterrupt: 

In [ ]:
int(len(history)/10)
